# Simulation Studies with SimulationRunner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/03_simulation_studies.ipynb)

`SimulationRunner` is the research-oriented layer of `catsim`. It runs one
simulated CAT session on a `CatEngine` for each true ability it is given, keeps
run-level exposure counts, and returns a `SimulationResult` with aggregate
metrics.

In [ ]:
import numpy as np

from catsim.engine import CatEngine
from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer
from catsim.item_bank import ItemBank
from catsim.selection import LinearSelector, MaxInfoSelector, RandomSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import MaxLengthStopper

In [ ]:
item_bank = ItemBank.generate(150, rng=31)
true_thetas = np.linspace(-2.0, 2.0, 40)


def fmt(value: float | None) -> float | str:
  """Round a metric for printing; overlap_rate is None when test lengths differ."""
  return "n/a" if value is None else round(float(value), 3)

## A baseline simulation

In [ ]:
engine = CatEngine(
  item_bank=item_bank,
  initializer=FixedPointInitializer(0.0),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=MaxLengthStopper(12),
)
runner = SimulationRunner(engine)
baseline = runner.run(true_thetas, rng=99)

print("Bias:", fmt(baseline.bias))
print("MSE:", fmt(baseline.mse))
print("RMSE:", fmt(baseline.rmse))
print("Overlap rate:", fmt(baseline.overlap_rate))

## Drawing a population and seeding a run

`run` takes the examinees' true abilities; it never draws them. To simulate a
population rather than a grid, draw the abilities with a generator of your own,
so the same population can be passed to every strategy you compare. The run's
own random draws (the initializer, the selector's tie-breaking and the simulated
responses) come from one generator that `run` builds from `rng` at every call,
so the same `rng` repeats a run exactly and another `rng` gives other tests.

In [ ]:
population = np.random.default_rng(7).normal(0, 1, 40)
first = runner.run(population, rng=99)
again = runner.run(population, rng=99)
other = runner.run(population, rng=100)

print("Same rng, same tests:", first.administered_items == again.administered_items)
print("Other rng, same tests:", first.administered_items == other.administered_items)
print("RMSE with rng=99 and rng=100:", fmt(first.rmse), fmt(other.rmse))

## Compare a few strategies

Here we compare one adaptive selector, one random selector, and one linear
baseline under the same item bank and explicit theta values. One run is not
enough to rank strategies, so each selector runs with five seeds; seed 99
repeats the baseline above.

In [ ]:
strategies = {
  "max_info": MaxInfoSelector(),
  "random": RandomSelector(),
  "linear": LinearSelector(list(range(12))),
}
seeds = [99, 100, 101, 102, 103]
SEARCH_BOUND = 6.0  # NumericalSearchEstimator looks for theta in [-6, 6]

for name, selector in strategies.items():
  strategy_engine = CatEngine(
    item_bank=item_bank,
    initializer=FixedPointInitializer(0.0),
    selector=selector,
    estimator=NumericalSearchEstimator(),
    stopper=MaxLengthStopper(12),
  )
  results = [SimulationRunner(strategy_engine).run(true_thetas, rng=seed) for seed in seeds]
  print({
    "selector": name,
    "rmse_by_seed": [fmt(result.rmse) for result in results],
    "mean_rmse": fmt(np.mean([result.rmse for result in results])),
    "estimates_at_bound": sum(
      int(np.isclose(np.abs(result.final_thetas), SEARCH_BOUND, atol=0.01).sum()) for result in results
    ),
    "overlap_by_seed": [fmt(result.overlap_rate) for result in results],
  })

## Interpretation notes

- `bias` shows systematic over- or under-estimation.
- `mse` and `rmse` summarize accuracy.
- `max_info` has the lowest mean RMSE, but not with every seed, and `random`
  and `linear` swap places from one seed to the next: compare strategies over
  several replications rather than from a single run.
- Much of that noise comes from `NumericalSearchEstimator`, a
  maximum-likelihood estimator. On items that are poorly targeted to the
  examinee, the likelihood can peak at the edge of the search interval, so the
  estimate lands on -6 or 6 (`estimates_at_bound`), and a few such sessions
  dominate a run's RMSE. A Bayesian estimator such as `EAPEstimator` avoids
  this, because its prior pulls extreme estimates toward the prior mean.
- `overlap_rate` barely changes with the seed: it reflects how the selector
  reuses items, not luck.
- `overlap_rate` is defined only when every session has the same test length.
  With a variable-length rule, such as a `StopRule` around a
  `StandardErrorStopper`, it is `None`, which the cells above print as `n/a`.
- `sessions` and `exposure_rates` let you move from aggregate metrics back to
  individual CAT behavior.

In methodological studies, compare strategies on the same true abilities: draw
the population once and pass it to every run.